# Lab 2 — Precision Recruiting with Recommender Systems

**Mission:** Limited recruiter time means we cannot visit every school. Build a transparent system that answers:

1. **WHERE should we focus?** Rank schools by downstream value and operational feasibility.
2. **WHAT should we try there?** Recommend an engagement using patterns from behaviorally similar schools.

You will deliberately begin with the wrong objective, watch the ranking change, and then infer a promising action Jefferson High has never tried.

**Estimated time:** 75 minutes (Lab A: 30; Lab B: 45)

> **Use your coding assistant as a teammate.** Give it the current cell, the self-check output, and the goal. Ask it to explain the smallest useful change rather than rewriting the notebook.

Suggested prompt:

> I am working in a classroom Jupyter notebook. Explain what this self-check is testing, then suggest the smallest edit to the marked variables. Do not change the data or the test.

In [ ]:
from IPython.display import display, Markdown

def check(name, condition, hint=""):
    try:
        passed = bool(condition)
    except Exception as exc:
        passed = False
        hint = f"{hint} ({type(exc).__name__}: {exc})"
    icon = "✅" if passed else "❌"
    print(f"{icon} {name}")
    if not passed and hint:
        print(f"   Hint: {hint}")
    return passed

def mission_header(text):
    display(Markdown(f"> **Mission checkpoint:** {text}"))

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
pd.set_option("display.max_columns", 30)
pd.options.display.float_format = "{:,.3f}".format

## Prepared artifacts from Lab 1

Lab 2 uses validated CSV artifacts produced by the Lab 1 pipeline. Prepared copies are supplied so this lab remains runnable even if a team has not completed Lab 1. All records are fictional classroom data; protected characteristics are not used.

In [ ]:
schools = pd.read_csv("https://usard-demo.netlify.app/data/school_summary.csv")
engagements = pd.read_csv("https://usard-demo.netlify.app/data/clean_recruiting_events.csv", parse_dates=["event_date"])

print(f"Loaded {len(schools)} school summaries and {len(engagements)} clean events.")
schools.head(8)

# Lab A — WHERE should we focus?

## A1. Rank by activity

If appointments are the goal, Lincoln appears to win. Pause before running the next cell: is appointment volume the outcome USARD actually wants to optimize?

In [ ]:
appointment_ranking = schools.nlargest(5, "appointments")[["school_name", "appointments", "qualified", "contracts"]]
appointment_ranking

## A2. Look downstream

Complete the four marked column choices. The cell runs even before it is correct; the checks tell you what to fix.

In [ ]:
QUALIFIED_NUMERATOR = "appointments"   # TODO
QUALIFIED_DENOMINATOR = "appointments" # TODO
EFFICIENCY_NUMERATOR = "appointments"  # TODO
EFFICIENCY_DENOMINATOR = "recruiter_hours"

schools["qualified_rate"] = (
    schools[QUALIFIED_NUMERATOR] / schools[QUALIFIED_DENOMINATOR]
)
schools["contracts_per_hour"] = (
    schools[EFFICIENCY_NUMERATOR] / schools[EFFICIENCY_DENOMINATOR]
)

schools.nlargest(5, "contracts_per_hour")[[
    "school_name", "appointments", "qualified", "contracts", "contracts_per_hour"
]]

In [ ]:
check("Qualification rate uses qualified ÷ appointments",
      QUALIFIED_NUMERATOR == "qualified" and QUALIFIED_DENOMINATOR == "appointments",
      "The numerator is the number that made it through qualification.")
check("Efficiency uses contracts ÷ recruiter hours",
      EFFICIENCY_NUMERATOR == "contracts" and EFFICIENCY_DENOMINATOR == "recruiter_hours",
      "We care about downstream success per constrained hour.")
check("Lincoln's appointment volume does not make it the efficiency leader",
      schools.nlargest(1, "appointments").iloc[0]["school_name"] != schools.nlargest(1, "contracts_per_hour").iloc[0]["school_name"],
      "Fix the efficiency numerator first.")

## A3. Define an explainable opportunity score

The mission owner chose:

- 60% downstream efficiency
- 25% qualification rate
- 15% school access

Edit the three weights. The algorithm cannot decide the objective for us.

In [ ]:
scaler = MinMaxScaler()
features = ["contracts_per_hour", "qualified_rate", "access_score"]
schools[["success_norm", "qualified_norm", "access_norm"]] = scaler.fit_transform(schools[features])

SUCCESS_WEIGHT = .34   # TODO
QUALIFIED_WEIGHT = .33 # TODO
ACCESS_WEIGHT = .33    # TODO

schools["opportunity_score"] = (
    SUCCESS_WEIGHT * schools["success_norm"]
    + QUALIFIED_WEIGHT * schools["qualified_norm"]
    + ACCESS_WEIGHT * schools["access_norm"]
)

In [ ]:
check("Weights sum to 1", np.isclose(SUCCESS_WEIGHT + QUALIFIED_WEIGHT + ACCESS_WEIGHT, 1.0))
check("Weights match the mission objective",
      np.allclose([SUCCESS_WEIGHT, QUALIFIED_WEIGHT, ACCESS_WEIGHT], [.60, .25, .15]),
      "Translate 60%, 25%, and 15% into decimals.")

## A4. Filter infeasible or thinly supported options

Scores do not override operations. Set the thresholds to **30 miles** and at least **4 historical events**. Event count is observable evidence volume—not a made-up “data quality” score.

In [ ]:
MAX_DISTANCE = 999       # TODO
MIN_HISTORICAL_EVENTS = 0 # TODO

eligible = schools.loc[
    schools["distance_miles"].le(MAX_DISTANCE)
    & schools["historical_events"].ge(MIN_HISTORICAL_EVENTS)
].copy()

excluded = schools.loc[~schools.index.isin(eligible.index), [
    "school_name", "distance_miles", "historical_events", "opportunity_score"
]].sort_values("opportunity_score", ascending=False)
display(Markdown("**Excluded options**"))
display(excluded)

In [ ]:
check("Distance threshold is operationally correct", MAX_DISTANCE == 30)
check("Evidence threshold is operationally correct", MIN_HISTORICAL_EVENTS == 4)
check("Liberty is excluded for travel", "Liberty High" not in set(eligible["school_name"]))
check("Victory is excluded for insufficient history", "Victory High" not in set(eligible["school_name"]))
check("No arbitrary data-quality metric is used", "data_quality" not in schools.columns)

## A5. Return Top K

Set `K = 5`. The winning school at the top of this list becomes the target for Lab B.

In [ ]:
K = 3  # TODO
top_schools = eligible.nlargest(K, "opportunity_score").copy()
top_schools[["school_name", "opportunity_score", "contracts_per_hour", "qualified_rate", "access_score"]]

In [ ]:
check("Top K returns five schools", K == 5 and len(top_schools) == 5)
check("Jefferson wins the final school ranking", top_schools.iloc[0]["school_name"] == "Jefferson High")

ax = top_schools.sort_values("opportunity_score").plot.barh(
    x="school_name", y="opportunity_score", legend=False, color="#1f5a91", figsize=(8, 4)
)
ax.set(title="Recommended schools after scoring and constraints", xlabel="Opportunity score", ylabel="")
plt.tight_layout()
plt.show()

> **Aha:** A recommender can produce a perfectly correct ranking for the wrong objective. There is no universal “best” school—only a ranking tied to an objective, evidence, and constraints.

# Lab B — WHAT should we do there?

Lab A selected Jefferson High. We now treat schools like “users,” engagement actions like “items,” and historical contracts per recruiter-hour like a “rating” to decide what to try there.

In [ ]:
actions = [
    "Cyber Careers Event", "STEM Careers Presentation", "Mechanical Careers Demo",
    "Healthcare Careers Session", "Education Benefits Session", "General Recruiting Table"
]

print(f"The event artifact contains {len(engagements):,} validated engagements.")
engagements.sample(8, random_state=SEED)[[
    "engagement_id", "event_date", "school_name", "action",
    "recruiter_hours", "contracts"
]]

## B1. Build the school × action matrix

What does the blank cell for Jefferson + Mechanical mean? It means **unobserved**, not failed.

In [ ]:
school_action_summary = (
    engagements
    .groupby(["school_name", "action"])
    .agg(
        total_hours=("recruiter_hours", "sum"),
        total_contracts=("contracts", "sum"),
        event_count=("engagement_id", "count"),
    )
)
school_action_summary["effectiveness"] = (
    school_action_summary["total_contracts"]
    / school_action_summary["total_hours"]
)

school_action = (
    school_action_summary["effectiveness"]
    .unstack()
    .reindex(columns=actions)
)
event_counts = (
    school_action_summary["event_count"]
    .unstack()
    .reindex(columns=actions)
)

display(Markdown("**Contracts per recruiter-hour**"))
display(school_action.style.format("{:.2f}", na_rep="—").background_gradient(cmap="Blues", axis=None))
display(Markdown("**Historical event count behind each score**"))
display(event_counts.style.format("{:.0f}", na_rep="—"))

In [ ]:
TARGET_SCHOOL = top_schools.iloc[0]["school_name"]
check("Lab B follows Lab A's winning school", TARGET_SCHOOL == "Jefferson High")
check("Jefferson has no Mechanical history", pd.isna(school_action.loc[TARGET_SCHOOL, "Mechanical Careers Demo"]))
check("Jefferson does have Healthcare history", pd.notna(school_action.loc[TARGET_SCHOOL, "Healthcare Careers Session"]))
check("Missing does not become zero", not (school_action.fillna(-1).loc[TARGET_SCHOOL, "Mechanical Careers Demo"] == 0))

## B2. Find behaviorally similar schools

Cosine similarity should use only actions observed at both schools. Require at least **three** overlapping actions so a one- or two-action coincidence cannot dominate the neighborhood.

In [ ]:
MIN_OVERLAP = 1  # TODO

def cosine_on_overlap(a, b, min_overlap=MIN_OVERLAP):
    mask = a.notna() & b.notna()
    if mask.sum() < min_overlap:
        return np.nan
    x = a[mask].to_numpy(dtype=float)
    y = b[mask].to_numpy(dtype=float)
    denominator = np.linalg.norm(x) * np.linalg.norm(y)
    return np.nan if denominator == 0 else float(np.dot(x, y) / denominator)

target_vector = school_action.loc[TARGET_SCHOOL]
overlap_counts = pd.Series({
    school: int((target_vector.notna() & school_action.loc[school].notna()).sum())
    for school in school_action.index if school != TARGET_SCHOOL
}, name="overlap_count")
similarities = pd.Series({
    school: cosine_on_overlap(target_vector, school_action.loc[school])
    for school in school_action.index if school != TARGET_SCHOOL
}, name="similarity").dropna().sort_values(ascending=False)

similarity_table = pd.concat([similarities, overlap_counts], axis=1).dropna().sort_values("similarity", ascending=False)
similarity_table

In [ ]:
check("Similarity requires at least three overlaps", MIN_OVERLAP == 3,
      "One or two shared actions are too little evidence for a stable neighborhood.")
check("Washington is Jefferson's closest behavioral neighbor", similarities.index[0] == "Washington High")
check("The neighborhood is meaningfully spread out", similarities.median() < .90,
      "Require three overlaps and verify that the school profiles are not all pointing in nearly the same direction.")

## B3. Predict an untried action

Turn on similarity weighting. A close neighbor should contribute more than a weak neighbor. To avoid diluting the signal with every weakly related school, use the three nearest behavioral neighbors.

In [ ]:
USE_SIMILARITY_WEIGHTS = False  # TODO
NEIGHBOR_COUNT = 3

def predict_action(action, matrix=school_action, sims=similarities):
    evidence = []
    for school, similarity in sims.head(NEIGHBOR_COUNT).items():
        value = matrix.loc[school, action]
        if pd.notna(value) and similarity > 0:
            evidence.append((school, float(similarity), float(value)))
    if not evidence:
        return np.nan
    if USE_SIMILARITY_WEIGHTS:
        numerator = sum(sim * value for _, sim, value in evidence)
        denominator = sum(sim for _, sim, _ in evidence)
        return numerator / denominator
    return np.mean([value for _, _, value in evidence])

mechanical_prediction = predict_action("Mechanical Careers Demo")
print(f"Predicted contracts per recruiter-hour: {mechanical_prediction:.3f}")

In [ ]:
check("Prediction uses similarity weighting", USE_SIMILARITY_WEIGHTS)
check("Mechanical is predicted to be promising", mechanical_prediction > .60,
      "Verify the similarity-weighted average and the target's missing cell.")

## B4. Rank observed and predicted actions together

Keep provenance visible. A predicted score is not the same kind of evidence as an observed score.

In [ ]:
recommendations = school_action.loc[TARGET_SCHOOL].copy()
evidence_type = pd.Series("observed", index=recommendations.index)

for action in recommendations.index[recommendations.isna()]:
    recommendations[action] = predict_action(action)
    evidence_type[action] = "predicted"

action_ranking = pd.DataFrame({
    "score": recommendations,
    "evidence": evidence_type,
}).sort_values("score", ascending=False)

action_ranking.head(3)

In [ ]:
check("Mechanical is the top recommendation", action_ranking.index[0] == "Mechanical Careers Demo")
check("Mechanical is labeled predicted", action_ranking.loc["Mechanical Careers Demo", "evidence"] == "predicted")
originally_observed = school_action.loc[TARGET_SCHOOL].dropna().index
originally_missing = school_action.loc[TARGET_SCHOOL].index[school_action.loc[TARGET_SCHOOL].isna()]
check("Observed actions remain labeled observed", (evidence_type.loc[originally_observed] == "observed").all())
check("Every filled blank is labeled predicted", (evidence_type.loc[originally_missing] == "predicted").all())

## Optional challenge — Content and hybrid evidence

Collaborative evidence asks, “What worked at schools that behaved like Jefferson?” Content evidence asks, “What fits Jefferson’s aggregate program profile?” Combine them only if you can explain the weights.

In [ ]:
dimensions = ["cyber", "engineering", "mechanical", "healthcare", "education"]
jefferson_profile = np.array([[.90, .80, .65, .15, .40]])
action_profiles = pd.DataFrame([
    [.95, .75, .20, .05, .15], [.70, .95, .35, .05, .20], [.10, .60, 1.0, .00, .20],
    [.05, .10, .05, 1.0, .15], [.10, .20, .10, .15, 1.0], [.35, .35, .35, .35, .35],
], index=actions, columns=dimensions)

content_scores = pd.Series(
    cosine_similarity(jefferson_profile, action_profiles.values)[0], index=actions, name="content_score"
)
collab_norm = (recommendations - recommendations.min()) / (recommendations.max() - recommendations.min())
hybrid = pd.DataFrame({"collaborative": collab_norm, "content": content_scores})
hybrid["hybrid"] = .60 * hybrid["collaborative"] + .40 * hybrid["content"]
hybrid.sort_values("hybrid", ascending=False).head(3)

## Red-team pause

Discuss before deployment:

- Are we learning what works—or what recruiters historically chose to try?
- How old can evidence be before it becomes stale?
- Should a prediction based on two overlapping actions receive the same confidence as one based on five?
- Which fields must never be used as ranking features?

**Transition:** We now have a WHERE and a WHAT. The recruiter still needs authoritative information before acting. That is the job of RAG.